# m04_l02_agent_loop

Exobean Engineering. Run the cells from top to bottom.

In [ ]:
# Google Colab only: fetches the free code companion, then run the rest.
import io, os, sys, urllib.request, zipfile
URL = "https://github.com/exobean/ai-agents-level1/releases/latest/download/shopassist-companion.zip"
if "google.colab" in sys.modules and not os.path.isdir("shopassist"):
    try:
        data = io.BytesIO(urllib.request.urlopen(URL).read())
    except OSError:  # network blocks the download: upload the zip
        from google.colab import files
        data = next(iter(files.upload()))
    zipfile.ZipFile(data).extractall(".")
    os.chdir(next(d for d in os.listdir(".") if d.endswith("-companion")))
    sys.path.insert(0, os.getcwd())


## Run

In [ ]:
import textwrap

from shopassist.agent import run_agent
from shopassist.models import TeachingModel
from shopassist.tools import GET_ORDER

SYSTEM = "You are ShopAssist, a polite support agent."
question = "Where is my order #1042? Can I still change the address?"

model = TeachingModel()
answer, messages = run_agent(model, [GET_ORDER], SYSTEM, question)

for i, m in enumerate(messages, 1):
    if m.tool_calls:
        c = m.tool_calls[0]
        body = f"wants tool {c.name}({c.args})"
    else:
        body = m.text
    for j, line in enumerate(textwrap.wrap(body, 52)):
        tag = f"{i}. {m.role:<9}" if j == 0 else ""
        print(f"{tag:<13}{line}")


## Runaway

In [ ]:
from shopassist.agent import StepLimitReached, run_agent
from shopassist.messages import Message, ToolCall
from shopassist.models import ScriptedModel
from shopassist.tools import GET_ORDER

# A model stuck in a rut: it asks for the same tool again and again.
again = ToolCall("c1", "get_order", {"order_id": "1042"})
stuck = ScriptedModel([Message("assistant", tool_calls=(again,))])

try:
    run_agent(stuck, [GET_ORDER], "You are ShopAssist.", "#1042?",
              max_steps=5)
except StepLimitReached as err:
    print("stopped:", err)
print("model calls made:", stuck.calls)


## Tool error

In [ ]:
import textwrap

from shopassist.agent import run_agent
from shopassist.models import TeachingModel
from shopassist.tools import GET_ORDER

question = "Where is order #9999?"
answer, messages = run_agent(TeachingModel(), [GET_ORDER],
                             "You are ShopAssist.", question)

result = messages[2]                 # what the tool sent back
print("tool error?  ", result.is_error)
print("tool said:   ", result.text)
print(textwrap.fill(answer, 60, initial_indent="agent answer: ",
                    subsequent_indent=" " * 14))


## Wire

In [ ]:
import json

from shopassist.agent import run_agent
from shopassist.models import TeachingModel
from shopassist.providers import to_request
from shopassist.tools import GET_ORDER

SYSTEM = "You are ShopAssist."
_, messages = run_agent(TeachingModel(), [GET_ORDER], SYSTEM, "#1042?")

# The JSON a real provider would receive on the second model call:
body = to_request(SYSTEM, messages[:3], [GET_ORDER])
body["tools"] = ["...same tool definition as before..."]
result = body["messages"][2]["content"][0]
result["content"] = result["content"][:24] + "...(shortened)"
print(json.dumps(body, indent=1))
